# Session 7: Vertical resolution, vertical errors and aircraft mass

Sessions 4–6 found that climbing above the moist layer (FL370 and above) removed
contrail exposure in every scenario tested, and that the minimum-fuel route was
already contrail-free. Both findings have untested weak points:

1. **Vertical resolution.** FL370 and FL380 lie between the 225 and 200 hPa ERA5
   levels, where values were linearly interpolated across the tropopause.
2. **Vertical errors.** No scenario moved the moist layer *up or down*. Scaling
   changed how humid it is; displacement moved it sideways or in time.
3. **Aircraft mass.** The PS model estimated the mass. A heavier aircraft has a
   lower best flight level, which could put the minimum-fuel route in the moist layer.

**This session:**
- **A.** Reload ERA5 on 10 pressure levels (175–300 hPa, every 10 hPa between 190
  and 250) and compare with the 5-level data. Adds FL390, the A320's highest
  practical cruise level.
- **B.** Move the whole field up or down by 150, 300 and 600 m and re-evaluate
  the FL340–FL390 routes.
- **C.** Recompute fuel for start-of-cruise masses of 56 t, the model estimate
  (≈64 t), 68 t and 72 t, and flag routes outside the aircraft's flight envelope.
- **D.** Re-run the deterministic vs robust route choice for each mass, across
  the vertical scenarios.

From this session on, reusable logic lives in `src/contrail_routing/`
(`config`, `scenarios`, `performance`, `decisions`); this notebook only calls it.

In [2]:
import sys, warnings
from pathlib import Path
SRC = (Path("..") / "src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pycontrails.physics import thermo, units

from contrail_routing import config
from contrail_routing.met import load_era5, check_met_box, load_region
from contrail_routing.scenarios import shift_field, coverage, run_scenarios, pcr_by_scenario
from contrail_routing.performance import fuel_table
from contrail_routing.decisions import choose, pareto_with_marginal

warnings.filterwarnings("ignore", message=r"\s*Met data appears to have originated from ECMWF")

results_dir = Path("..") / "results"
fig_dir = results_dir / "figures"
s7_dir = results_dir / "session7"
s7_dir.mkdir(parents=True, exist_ok=True)

FLIGHT_LEVELS_7 = config.FLIGHT_LEVELS + [390]
specs_all = config.route_specs(flight_levels=FLIGHT_LEVELS_7)
specs_df = pd.DataFrame.from_dict(specs_all, orient="index")
routes_all = config.build_routes(specs_all)
print(len(routes_all), "routes: flight levels", FLIGHT_LEVELS_7, "x offsets", config.OFFSETS_KM)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
45 routes: flight levels [310, 320, 330, 340, 350, 360, 370, 380, 390] x offsets [-300, -150, 0, 150, 300]


In [3]:
from pycontrails.datalib.ecmwf import ERA5ARCO

era5_fine = ERA5ARCO(time=config.ERA5_TIME,
                     variables=["air_temperature", "specific_humidity"],
                     pressure_levels=config.LEVELS_FINE)
bad = Path(era5_fine.create_cachepath(pd.Timestamp("2022-03-01T02")))
print("Deleting:", bad, "| exists:", bad.exists())
bad.unlink()
print("Still exists:", bad.exists())

Deleting: C:\Users\barqb\AppData\Local\pycontrails\pycontrails\Cache\arcoera5-e75aaac55d4e72d0a032c7abcad3ac02.nc | exists: True
Still exists: False


## A. Load the data on coarse and fine pressure levels

The coarse (5-level) data loads from the cache built in Session 2. The fine
(10-level) data is new: **the first run downloads and converts it hour by hour
and can take 10–20 minutes**. Close other heavy programs (especially browsers)
while it runs. After that it loads from the cache in seconds.

In [4]:
met_coarse = load_era5(time=config.ERA5_TIME, pressure_levels=config.LEVELS_COARSE)
met_fine = load_era5(time=config.ERA5_TIME, pressure_levels=config.LEVELS_FINE)
print(check_met_box(met_fine, config.REGION_LON, config.REGION_LAT))

region_coarse = load_region(met_coarse, config.REGION_LON, config.REGION_LAT)
region_fine = load_region(met_fine, config.REGION_LON, config.REGION_LAT)
print("Coarse levels (hPa):", region_coarse.data["level"].values)
print("Fine levels (hPa):  ", region_fine.data["level"].values)

                 time  air_temperature_nan  specific_humidity_nan
0 2022-03-01 00:00:00                    0                      0
1 2022-03-01 01:00:00                    0                      0
2 2022-03-01 02:00:00                    0                      0
3 2022-03-01 03:00:00                    0                      0
4 2022-03-01 04:00:00                    0                      0
5 2022-03-01 05:00:00                    0                      0
6 2022-03-01 06:00:00                    0                      0
Coarse levels (hPa): [200. 225. 250. 275. 300.]
Fine levels (hPa):   [175. 190. 200. 210. 220. 230. 240. 250. 275. 300.]


In [ ]:
cov_coarse = coverage(region_coarse, routes_all)
cov_fine = coverage(region_fine, routes_all)
print("Routes outside the coarse data:", cov_coarse.index[~cov_coarse["ok"]].tolist())
assert cov_fine["ok"].all(), "Some routes are outside the fine data"

routes_coarse = {n: r for n, r in routes_all.items() if cov_coarse.at[n, "ok"]}
res_base = run_scenarios({"coarse": region_coarse}, routes_coarse, s7_dir, "base")
res_base |= run_scenarios({"fine": region_fine}, routes_all, s7_dir, "base")
pcr_res = pcr_by_scenario(res_base, specs_all)

# Regression: the coarse region must reproduce Session 4 exactly
m4 = pd.read_csv(results_dir / "session4_route_metrics.csv").set_index("route")
diff = (pcr_res["coarse"].dropna() - m4["pcr_km"]).abs().max()
print(f"\nMax |coarse - Session 4| pcr_km: {diff:.6f}")
assert diff < 1e-6
print("Coarse data reproduces Session 4.")

pcr_res = pcr_res.join(specs_df[["flight_level", "offset_km"]])
for col in ["coarse", "fine"]:
    print(f"\nDistance in PCR (km), {col} levels:")
    print(pcr_res.pivot(index="flight_level", columns="offset_km", values=col).round(0))
pcr_res.to_csv(results_dir / "session7_pcr_coarse_vs_fine.csv")

In [ ]:
def issr_profile(region):
    """Fraction of the region (all hours) that is ice-supersaturated, and max RHi, by level."""
    d = region.data
    temp = d["air_temperature"].transpose("longitude", "latitude", "level", "time").values
    q = d["specific_humidity"].transpose("longitude", "latitude", "level", "time").values
    p = d["level"].values[None, None, :, None] * 100.0
    rhi = thermo.rhi(q, temp, p)
    return pd.DataFrame({"issr_fraction": (rhi > 1.0).mean(axis=(0, 1, 3)),
                         "rhi_max": rhi.max(axis=(0, 1, 3))},
                        index=pd.Index(d["level"].values, name="level_hPa"))

prof_coarse, prof_fine = issr_profile(region_coarse), issr_profile(region_fine)
print("Coarse:\n", prof_coarse.round(3), "\n\nFine:\n", prof_fine.round(3))

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot(prof_fine["issr_fraction"], prof_fine.index, "o-", label="10 levels")
ax.plot(prof_coarse["issr_fraction"], prof_coarse.index, "s--", label="5 levels")
for fl in [340, 360, 370, 380, 390]:
    p_fl = float(units.m_to_pl(units.ft_to_m(fl * 100.0)))
    ax.axhline(p_fl, color="grey", linewidth=0.8, linestyle=":")
    ax.annotate(f"FL{fl}", (ax.get_xlim()[1], p_fl), xytext=(-30, 2),
                textcoords="offset points", fontsize=8, color="grey")
ax.invert_yaxis()
ax.set_xlabel("Fraction of region ice-supersaturated (all hours)")
ax.set_ylabel("Pressure (hPa)")
ax.set_title("Where the moist layer ends: 5 vs 10 ERA5 levels")
ax.legend()
ax.grid(alpha=0.3)
fig.savefig(fig_dir / "session7_issr_profile_coarse_vs_fine.png", dpi=150, bbox_inches="tight")
plt.show()

## B. Vertical displacement of the whole field

Pressure levels are relabelled so the entire temperature and humidity field moves
up or down (`shift_field(up_m=...)`). Only FL340–FL390 are evaluated: the
largest shifts move the data's lowest level above FL310–FL330.

In [ ]:
VERTICAL_M = {"base": 0.0, "up150": 150.0, "up300": 300.0, "up600": 600.0,
              "down150": -150.0, "down300": -300.0, "down600": -600.0}
specs_v = {n: s for n, s in specs_all.items() if s["flight_level"] >= 340}
routes_v = {n: routes_all[n] for n in specs_v}

scen_v = {name: shift_field(region_fine, up_m=dz) for name, dz in VERTICAL_M.items()}
for name, met in scen_v.items():
    cov = coverage(met, routes_v)
    assert cov["ok"].all(), f"{name}: routes outside shifted data: {cov.index[~cov['ok']].tolist()}"
print("All shifted fields cover all FL340-FL390 routes.")

res_v = run_scenarios(scen_v, routes_v, s7_dir, "vertical")
pcr_v = pcr_by_scenario(res_v, specs_all)
assert np.allclose(pcr_v["base"], pcr_res.loc[pcr_v.index, "fine"])
print("Zero shift reproduces the fine-level results.")

key_v = ["GC0_FL390", "GC0_FL380", "GC0_FL370", "GC0_FL360", "NW150_FL360",
         "NW300_FL360", "GC0_FL350", "GC0_FL340", "NW150_FL340"]
print("\nPCR km under vertical shifts (key routes):")
print(pcr_v.loc[key_v].round(1).to_string())
always_clean = pcr_v.index[pcr_v.max(axis=1) == 0].tolist()
print("\nContrail-free in every vertical scenario:", always_clean)
print("Contrail-free in base but not in every scenario:",
      pcr_v.index[(pcr_v["base"] == 0) & (pcr_v.max(axis=1) > 0)].tolist())
pcr_v.to_csv(results_dir / "session7_pcr_vertical_shifts.csv")

## C. Fuel and aircraft mass

Fuel is recomputed with the PS model for each start-of-cruise mass, using the
fine-level temperatures. The A320 in the PS model has a maximum take-off mass
of 73.5 t, so 72 t is close to a full aircraft at the start of cruise. Routes
where the aircraft is heavier than its buffet-limited maximum mass for that
level are marked infeasible.

In [ ]:
MASSES = {"light_56t": 56000.0, "model_default": None,
          "heavy_68t": 68000.0, "very_heavy_72t": 72000.0}
fuel_long = fuel_table(res_base["fine"], config.AIRCRAFT, config.CRUISE_SPEED_MS, MASSES)
fuel_long = fuel_long.join(specs_df[["flight_level", "offset_km"]], on="route")
fuel_long["feasible"] = fuel_long["within_envelope"] & fuel_long["fuel_kg"].notna()
print("Model-estimated start mass (kg):",
      fuel_long.loc[fuel_long["mass_case"] == "model_default", "start_mass_kg"].describe()[["min", "mean", "max"]].round(0).to_dict())

fuel_by_mass = fuel_long.pivot(index="route", columns="mass_case", values="fuel_kg")[list(MASSES)]
feasible = fuel_long.pivot(index="route", columns="mass_case", values="feasible")[list(MASSES)].astype(bool)

gc = fuel_long[fuel_long["offset_km"] == 0]
gc_fuel = gc.pivot(index="flight_level", columns="mass_case", values="fuel_kg")[list(MASSES)]
gc_ok = gc.pivot(index="flight_level", columns="mass_case", values="feasible")[list(MASSES)].astype(bool)
print("\nGreat-circle route: extra fuel vs that mass's best feasible level (kg); X = outside envelope")
print((gc_fuel - gc_fuel.where(gc_ok).min()).round(0).astype(int).astype(str).where(gc_ok, "X"))

rows = []
for mc in MASSES:
    f = fuel_by_mass[mc].where(feasible[mc])
    r = f.idxmin()
    rows.append({"mass_case": mc, "min_fuel_route": r, "fuel_kg": f[r],
                 "pcr_fine_km": pcr_res.at[r, "fine"],
                 "pcr_worst_vertical_km": pcr_v.loc[r].max() if r in pcr_v.index else np.nan,
                 "n_infeasible_routes": int((~feasible[mc]).sum())})
min_fuel_by_mass = pd.DataFrame(rows)
print("\nMinimum-fuel feasible route for each mass:")
print(min_fuel_by_mass.round(1).to_string(index=False))
fuel_long.to_csv(results_dir / "session7_fuel_by_mass.csv", index=False)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for mc in MASSES:
    ok = gc_ok[mc]
    rel = gc_fuel[mc] - gc_fuel[mc].where(ok).min()
    line, = ax1.plot(rel.index[ok], rel[ok], "o-", label=mc)
    ax1.plot(rel.index[~ok], rel[~ok], "x", color=line.get_color(), markersize=10)
ax1.set_xlabel("Flight level")
ax1.set_ylabel("Extra fuel vs best feasible level (kg)")
ax1.set_title("Great circle: fuel by flight level and mass (x = outside envelope)")
ax1.grid(alpha=0.3)
ax1.legend()

gcp = pcr_res[pcr_res["offset_km"] == 0].set_index("flight_level")["fine"]
ax2.bar(gcp.index, gcp.values, width=6)
for x, v in gcp.items():
    if v == 0:
        ax2.annotate("0", (x, 0), xytext=(0, 3), textcoords="offset points", ha="center")
ax2.set_xlabel("Flight level")
ax2.set_ylabel("Distance in PCR conditions (km)")
ax2.set_title("Great circle: contrail exposure by flight level (10-level data)")
ax2.grid(alpha=0.3, axis="y")
fig.savefig(fig_dir / "session7_fuel_by_mass_and_exposure.png", dpi=150, bbox_inches="tight")
plt.show()

## D. Route choice by mass, robust to vertical errors

For each mass, the candidates are the feasible FL340–FL390 routes. The three
decision rules from Sessions 5–6 are applied across the 7 vertical scenarios.
"Cheapest always-clean" is the lowest-fuel route that has zero exposure in every
vertical scenario, and its extra fuel is the price of guaranteed avoidance
(within these scenarios).

In [ ]:
LAMBDAS = [0.1, 1.0, 10.0]
decisions, summary = [], []
for mc in MASSES:
    fuel = fuel_by_mass[mc]
    cands = [r for r in routes_v if feasible.at[r, mc]]
    min_cand = fuel[cands].idxmin()
    clean = [r for r in cands if pcr_v.loc[r].max() == 0]
    best_clean = min(clean, key=lambda r: fuel[r]) if clean else None
    summary.append({
        "mass_case": mc, "n_candidates": len(cands),
        "min_fuel_route": min_cand, "its_pcr_base_km": pcr_v.at[min_cand, "base"],
        "its_pcr_worst_km": pcr_v.loc[min_cand].max(),
        "cheapest_always_clean": best_clean,
        "extra_fuel_for_always_clean_kg": (fuel[best_clean] - fuel[min_cand]) if best_clean else np.nan,
    })
    for lam in LAMBDAS:
        d = choose(pcr_v, fuel, cands, lam)
        d.insert(0, "lambda", lam)
        d.insert(0, "mass_case", mc)
        decisions.append(d)

summary = pd.DataFrame(summary)
decisions = pd.concat(decisions, ignore_index=True)
pd.set_option("display.width", 220)
print("Per mass (FL340-FL390 candidates, 7 vertical scenarios):")
print(summary.round(1).to_string(index=False))

disagree = decisions.groupby(["mass_case", "lambda"])["route"].transform("nunique") > 1
n_cases = decisions.groupby(["mass_case", "lambda"]).ngroups
n_dis = int(disagree.groupby([decisions["mass_case"], decisions["lambda"]]).any().sum())
print(f"\nCases where the rules disagree: {n_dis} of {n_cases}")
print(decisions[disagree].round(1).to_string(index=False))

summary.to_csv(results_dir / "session7_summary_by_mass.csv", index=False)
decisions.to_csv(results_dir / "session7_robust_decisions.csv", index=False)

## Session 7 results

*(Written after reviewing the outputs above.)*